# Tiny Text-to-Image Model with imagen-pytorch

This notebook experiments with `imagen-pytorch`, an open-source implementation of Google's Imagen text-to-image diffusion model. It builds a very small Imagen (one U-Net, T5-small text encoder), trains it briefly on CIFAR-10 images with captions made from the class names ("a photo of a cat"), and then samples images from text prompts. Expect blurry results: real text-to-image models need far more data and compute; this is for learning the API.

**How to run:** Open in Google Colab with *Runtime → Change runtime type → GPU* and run all cells. The first cell installs `imagen-pytorch`; CIFAR-10 downloads through torchvision.

_Rewritten from my original Colab notebook: `Untitled9.ipynb`._

In [ ]:
!pip install -q imagen-pytorch

## 1. Imports and seeds

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from imagen_pytorch import Unet, Imagen, ImagenTrainer

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Data: CIFAR-10 images with simple captions

Imagen expects images with pixel values in [0, 1], which is what `ToTensor()` produces.

In [ ]:
train_set = datasets.CIFAR10(root="data", train=True, download=True, transform=transforms.ToTensor())
class_names = train_set.classes
loader = DataLoader(train_set, batch_size=32, shuffle=True, drop_last=True)

## 3. Build a small Imagen

In [ ]:
unet = Unet(
    dim=32,
    cond_dim=512,
    dim_mults=(1, 2, 4),
    num_resnet_blocks=2,
    layer_attns=(False, False, True),
    layer_cross_attns=(False, False, True),
)

imagen = Imagen(
    unets=(unet,),
    text_encoder_name="t5-small",
    image_sizes=(32,),
    timesteps=250,
    cond_drop_prob=0.1,
)

trainer = ImagenTrainer(imagen).to(device)

## 4. Train for a few steps

Increase `NUM_STEPS` for better images (training is slow).

In [ ]:
NUM_STEPS = 1000
step = 0
while step < NUM_STEPS:
    for images, labels in loader:
        texts = [f"a photo of a {class_names[l]}" for l in labels]
        loss = trainer(images.to(device), texts=texts, unet_number=1)
        trainer.update(unet_number=1)
        if step % 100 == 0:
            print(f"step {step}: loss {loss:.4f}")
        step += 1
        if step >= NUM_STEPS:
            break

## 5. Generate images from text

In [ ]:
prompts = ["a photo of a ship", "a photo of a frog", "a photo of a horse", "a photo of a truck"]
samples = trainer.sample(texts=prompts, cond_scale=3.0)  # tensor (N, 3, 32, 32) in [0, 1]

grid = make_grid(samples.cpu().clamp(0, 1), nrow=len(prompts))
plt.figure(figsize=(8, 3))
plt.imshow(grid.permute(1, 2, 0))
plt.title(" | ".join(p.replace("a photo of a ", "") for p in prompts))
plt.axis("off")
plt.show()